# Painel final — lista de prioridade e valores de referência

**Squad Observatório do Clima – Grande Desafio DataConnect Solutions**

Este notebook gera **todos os números do painel e da apresentação** a partir do `dataset_mestre_observatorio.csv`, aplicando as decisões da célula (reunião de 19/10/2026):

| Decisão | Regra |
|---|---|
| Funil | SINISA respondido → moradias em risco > 0 → sem CAPAG A/B → risco mapeado → crítico e com dano no Atlas (318 elegíveis) |
| Lista oficial (A) | Os 20 com mais moradias em risco (desempate: desastres, depois pessoas afetadas) |
| Alternativa estudada (C) | Nota = 70% percentil do nº de moradias em risco + 30% percentil da parte da cidade em risco |
| Valores | Referência: mesmo R$ por moradia, piso R$ 3 mi, teto R$ 15 mi, total R$ 150 mi |
| Danos | Danos materiais + prejuízo econômico público (Atlas) |

**Saídas** (pasta `painel/`): os 3 CSVs que o Looker Studio usa (`painel_top20.csv`, `painel_funil.csv`, `painel_dentro_fora.csv`) e as tabelas de apoio (suplentes, alternativa C, trocas A × C, ranking dos 318).

## 0. Leitura

In [1]:
import os
import numpy as np
import pandas as pd

URL_MESTRE = 'https://raw.githubusercontent.com/Reginajose/Observatorio-Clima-Grande-desafio/main/data/dataset_mestre_observatorio.csv'
URL_COORD = 'https://raw.githubusercontent.com/kelvins/municipios-brasileiros/main/csv/municipios.csv'  # lat/long por código IBGE

# usa o arquivo local se o notebook estiver rodando dentro do repositório; senão baixa do GitHub
caminho = next((c for c in ['data/dataset_mestre_observatorio.csv', '../data/dataset_mestre_observatorio.csv'] if os.path.exists(c)), URL_MESTRE)
d = pd.read_csv(caminho, encoding='utf-8-sig')
for col in ['criterio_fiscal_priorizacao', 'tem_dano_atlas', 'participou_sinisa']:
    d[col] = d[col].astype(str).str.strip().str.upper().eq('TRUE')
d['danos_rs'] = d['soma_danos_materiais'] + d['soma_prejuizo_publico']   # danos = DM + PEPL
print(caminho, d.shape)

../data/dataset_mestre_observatorio.csv (5570, 21)


## 1. Funil: de 5.570 a 318 elegíveis

In [2]:
etapas = []
f = d.copy();                                                  etapas.append(('Todos os municípios', len(f)))
f = f[f.participou_sinisa];                                    etapas.append(('Responderam o SINISA 2024', len(f)))
f = f[f.qtd_domicilios_risco > 0];                             etapas.append(('Declaram moradias em risco', len(f)))
f = f[f.criterio_fiscal_priorizacao];                          etapas.append(('Sem CAPAG A ou B', len(f)))
f = f[f.mapeamento_area_risco == 'Sim'];                       etapas.append(('Têm o risco mapeado', len(f)))
f = f[(f.municipio_critico == 'Sim') & f.tem_dano_atlas];      etapas.append(('Críticos e com dano no Atlas', len(f)))
eleg = f.copy()
assert len(eleg) == 318, 'O funil deveria terminar em 318 elegíveis'
pd.DataFrame(etapas, columns=['etapa', 'municipios'])

,etapa,municipios
0,Todos os municípios,5570
1,Responderam o SINISA 2024,4845
2,Declaram moradias em risco,2238
3,Sem CAPAG A ou B,1031
4,Têm o risco mapeado,621
5,Críticos e com dano no Atlas,318


## 2. Ordem dos 20 e valores de referência

`ordenar` produz a lista oficial (A) e a alternativa (C). `alocar` acha, por bisseção, o valor por moradia `r` tal que a soma de `clip(r × moradias, piso, teto)` dê exatamente R$ 150 mi.

In [3]:
def ordenar(e, opcao):
    e = e.copy()
    if opcao == 'A':
        chave = ['qtd_domicilios_risco', 'total_desastres', 'danos_humanos']
    else:  # 'C'
        e['nota'] = 0.7 * e.qtd_domicilios_risco.rank(pct=True) + 0.3 * e.pct_domicilios_risco.rank(pct=True)
        chave = ['nota', 'qtd_domicilios_risco', 'total_desastres']
    return e.sort_values(chave, ascending=False).reset_index(drop=True)

def alocar(moradias, total=150e6, piso=3e6, teto=15e6):
    lo, hi = 0.0, teto / moradias.min()
    for _ in range(200):
        r = (lo + hi) / 2
        if (r * moradias).clip(piso, teto).sum() < total: lo = r
        else: hi = r
    v = (r * moradias).clip(piso, teto).round(2)
    livre = v.index[(v > piso) & (v < teto)][0]          # resíduo de arredondamento vai para o 1º proporcional
    v.loc[livre] += round(total - v.sum(), 2)
    return v, r

def top20(opcao):
    t = ordenar(eleg, opcao).head(20).copy()
    t['posicao'] = range(1, 21)
    t['valor_referencia'], r = alocar(t.qtd_domicilios_risco)
    return t, r

A, rA = top20('A')
Cc, rC = top20('C')
for nome, t, r in [('A', A, rA), ('C', Cc, rC)]:
    assert abs(t.valor_referencia.sum() - 150e6) < 0.005
    print(f'{nome}: R$ {r:,.2f} por moradia · {int((t.valor_referencia >= 15e6 - .01).sum())} no teto · '
          f'{int((t.valor_referencia <= 3e6 + .01).sum())} no piso · soma R$ {t.valor_referencia.sum():,.2f}')
A[['posicao', 'Nome', 'UF', 'qtd_domicilios_risco', 'valor_referencia']]

A: R$ 319.40 por moradia · 4 no teto · 3 no piso · soma R$ 150,000,000.00
C: R$ 432.38 por moradia · 4 no teto · 5 no piso · soma R$ 150,000,000.00


,posicao,Nome,UF,qtd_domicilios_risco,valor_referencia
0,1,São Gonçalo,RJ,175838.0,15000000.00
1,2,Campo Grande,MS,125000.0,15000000.00
2,3,Canoas,RS,70000.0,15000000.00
3,4,Campos dos Goytacazes,RJ,63000.0,15000000.00
4,5,Duque de Caxias,RJ,31070.0,9923699.34
5,6,Esteio,RS,30000.0,9581943.36
6,7,Blumenau,SC,28758.0,9185250.90
7,8,Belford Roxo,RJ,25500.0,8144651.86
8,9,São José,SC,25000.0,7984952.80
9,10,Ilhéus,BA,20000.0,6387962.24


## 3. Conferência dos números do painel

In [4]:
fora = eleg[~eleg.Cod_IBGE_7.isin(A.Cod_IBGE_7)]
conf = pd.Series({
    'Moradias em risco (20)': A.qtd_domicilios_risco.sum(),
    'Pessoas afetadas 2021–25': A.danos_humanos.sum(),
    'Desastres 2021–25': A.total_desastres.sum(),
    'Danos DM + PEPL (R$)': round(A.danos_rs.sum(), 2),
    'Moradias dos que ficam de fora': fora.qtd_domicilios_risco.sum(),
    'Parte da cidade em risco, média (%)': round(A.pct_domicilios_risco.mean(), 1),
})
assert conf['Moradias em risco (20)'] == 713211 and conf['Pessoas afetadas 2021–25'] == 341914
conf

Moradias em risco (20)                 7.132110e+05
Pessoas afetadas 2021–25               3.419140e+05
Desastres 2021–25                      1.570000e+02
Danos DM + PEPL (R$)                   8.080276e+08
Moradias dos que ficam de fora         3.033020e+05
Parte da cidade em risco, média (%)    2.540000e+01
dtype: float64

## 4. Arquivos do Looker Studio

In [5]:
OUT = 'painel'
os.makedirs(OUT, exist_ok=True)
geo = pd.read_csv(URL_COORD).set_index('codigo_ibge')

top = pd.DataFrame({
    'posicao': A.posicao,
    'municipio': A.Nome,
    'uf': A.UF,
    'rotulo': [f'{p}. {n} ({u})' for p, n, u in zip(A.posicao, A.Nome, A.UF)],
    'municipio_uf': [f'{n} ({u})' for n, u in zip(A.Nome, A.UF)],
    'cod_ibge': A.Cod_IBGE_7,
    'capag': A.capag.fillna('sem nota'),
    'moradias_em_risco': A.qtd_domicilios_risco.astype(int),
    'parte_da_cidade_em_risco': (A.pct_domicilios_risco / 100).round(4),   # fração → tipo Percentual no Looker
    'desastres_2021_25': A.total_desastres.astype(int),
    'pessoas_afetadas_2021_25': A.danos_humanos.astype(int),
    'danos_materiais': A.soma_danos_materiais.round(2),
    'prejuizo_economico_publico': A.soma_prejuizo_publico.round(2),
    'danos_total': A.danos_rs.round(2),
    'valor_referencia': A.valor_referencia.values,
})
top['rs_por_moradia'] = (top.valor_referencia / top.moradias_em_risco).round(2)
top['regra_valor'] = np.where(top.valor_referencia >= 15e6 - .01, 'teto', np.where(top.valor_referencia <= 3e6 + .01, 'piso', 'proporcional'))
top['lat_long'] = [f"{geo.loc[c, 'latitude']},{geo.loc[c, 'longitude']}" for c in top.cod_ibge]
top.to_csv(f'{OUT}/painel_top20.csv', index=False, encoding='utf-8')

ROT = [e[0] for e in etapas] + ['Prioridade (esta lista)']
fun = pd.DataFrame({'ordem': range(1, len(ROT) + 1), 'etapa': ROT, 'municipios': [e[1] for e in etapas] + [20]})
fun['destaque'] = np.where(fun.ordem == len(ROT), 'sim', 'nao')
fun.to_csv(f'{OUT}/painel_funil.csv', index=False, encoding='utf-8')

g_in, g_out = 'Recebem (20)', f'Ficam de fora ({len(fora)})'
pd.DataFrame([('Municípios', 1, g_in, 20), ('Municípios', 1, g_out, len(fora)),
              ('Moradias em risco', 2, g_in, int(A.qtd_domicilios_risco.sum())),
              ('Moradias em risco', 2, g_out, int(fora.qtd_domicilios_risco.sum()))],
             columns=['indicador', 'ordem', 'grupo', 'valor']).to_csv(f'{OUT}/painel_dentro_fora.csv', index=False, encoding='utf-8')
top.head()

,posicao,municipio,uf,rotulo,municipio_uf,cod_ibge,capag,moradias_em_risco,parte_da_cidade_em_risco,desastres_2021_25,pessoas_afetadas_2021_25,danos_materiais,prejuizo_economico_publico,danos_total,valor_referencia,rs_por_moradia,regra_valor,lat_long
0,1,São Gonçalo,RJ,1. São Gonçalo (RJ),São Gonçalo (RJ),3304904,C,175838,0.4000,4,624,27405589.64,3075978.70,30481568.34,15000000.00,85.31,teto,"-22.8268,-43.0634"
1,2,Campo Grande,MS,2. Campo Grande (MS),Campo Grande (MS),5002704,C,125000,0.3043,56,0,112000.00,150000.00,262000.00,15000000.00,120.00,teto,"-20.4486,-54.6295"
2,3,Canoas,RS,3. Canoas (RS),Canoas (RS),4304606,C,70000,0.4353,7,184680,6910085.69,7025897.78,13935983.47,15000000.00,214.29,teto,"-29.9128,-51.1857"
3,4,Campos dos Goytacazes,RJ,4. Campos dos Goytacazes (RJ),Campos dos Goytacazes (RJ),3301009,C,63000,0.2721,8,1142,26239841.06,0.00,26239841.06,15000000.00,238.10,teto,"-21.7622,-41.3181"
4,5,Duque de Caxias,RJ,5. Duque de Caxias (RJ),Duque de Caxias (RJ),3301702,C,31070,0.0827,6,1971,21962522.90,2692000.26,24654523.16,9923699.34,319.40,proporcional,"-22.7858,-43.3049"


## 5. Tabelas de apoio: suplentes, alternativa C e ranking dos 318

In [6]:
rankA, rankC = ordenar(eleg, 'A'), ordenar(eleg, 'C')

sup = rankA.iloc[20:25]
pd.DataFrame({'posicao': range(21, 26), 'municipio': sup.Nome.values, 'uf': sup.UF.values,
              'moradias_em_risco': sup.qtd_domicilios_risco.astype(int).values,
              'parte_da_cidade_em_risco_pct': sup.pct_domicilios_risco.round(1).values}
             ).to_csv(f'{OUT}/suplentes.csv', index=False, encoding='utf-8')

cols = {'posicao': 'posicao', 'Nome': 'municipio', 'UF': 'uf', 'qtd_domicilios_risco': 'moradias_em_risco',
        'pct_domicilios_risco': 'parte_da_cidade_em_risco_pct', 'danos_humanos': 'pessoas_afetadas_2021_25',
        'danos_rs': 'danos_total', 'valor_referencia': 'valor_referencia'}
Cc[list(cols)].rename(columns=cols).to_csv(f'{OUT}/alternativa_C_top20.csv', index=False, encoding='utf-8')

sa, sc = set(A.Nome), set(Cc.Nome)
trocas = pd.concat([A[A.Nome.isin(sa - sc)].assign(situacao='sai na alternativa'),
                    Cc[Cc.Nome.isin(sc - sa)].assign(situacao='entra na alternativa')])
trocas[['situacao', 'Nome', 'UF', 'qtd_domicilios_risco', 'pct_domicilios_risco', 'valor_referencia']].rename(columns=cols)\
    .to_csv(f'{OUT}/trocas_A_C.csv', index=False, encoding='utf-8')

def resumo(t):
    return {'moradias_em_risco': t.qtd_domicilios_risco.sum(), 'pessoas_afetadas_2021_25': t.danos_humanos.sum(),
            'danos_total': round(t.danos_rs.sum(), 2), 'parte_da_cidade_em_risco_media_pct': round(t.pct_domicilios_risco.mean(), 1)}
comp = pd.DataFrame({'lista_A': resumo(A), 'alternativa_C': resumo(Cc)})
comp['diferenca'] = comp.alternativa_C - comp.lista_A
comp.to_csv(f'{OUT}/comparacao_A_C.csv', encoding='utf-8')

rk = eleg[['Cod_IBGE_7', 'Nome', 'UF', 'qtd_domicilios_risco', 'pct_domicilios_risco']].copy()
rk['posicao_A'] = rk.Cod_IBGE_7.map(dict(zip(rankA.Cod_IBGE_7, range(1, len(eleg) + 1))))
rk['posicao_C'] = rk.Cod_IBGE_7.map(dict(zip(rankC.Cod_IBGE_7, range(1, len(eleg) + 1))))
rk.sort_values('posicao_A').to_csv(f'{OUT}/ranking_318.csv', index=False, encoding='utf-8')

print('Em comum entre A e C:', len(sa & sc), '· suplentes de A que entram na C:', sorted(set(sup.Nome) & (sc - sa)))
comp

Em comum entre A e C: 13 · suplentes de A que entram na C: ['Barreiros', 'Codó', 'Paraty']


,lista_A,alternativa_C,diferenca
moradias_em_risco,7.132110e+05,6.351470e+05,-7.806400e+04
pessoas_afetadas_2021_25,3.419140e+05,2.833770e+05,-5.853700e+04
danos_total,8.080276e+08,6.742352e+08,-1.337924e+08
parte_da_cidade_em_risco_media_pct,2.540000e+01,3.690000e+01,1.150000e+01


## 6. Baixar os arquivos (no Colab)

In [7]:
import shutil
shutil.make_archive('painel', 'zip', OUT)
try:
    from google.colab import files
    files.download('painel.zip')
except ImportError:
    print('Arquivos salvos em', os.path.abspath(OUT), ':', sorted(os.listdir(OUT)))

Arquivos salvos em /tmp/tmp.CqFig3sotL/Notebook/painel : ['alternativa_C_top20.csv', 'comparacao_A_C.csv', 'painel_dentro_fora.csv', 'painel_funil.csv', 'painel_top20.csv', 'ranking_318.csv', 'suplentes.csv', 'trocas_A_C.csv']
